← [Factor de certeza](04-factor-de-certeza.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Redes neuronales](06-redes-neuronales.ipynb) →

# 05 · Motor de inferencia

El motor es el estudiante que usa el libro (reglas) sobre la hoja de examen
(hechos). Reci combina **cuatro mecanismos**: encadenamiento hacia adelante,
encadenamiento hacia atrás, meta-reglas y validación.



## Estado vigente del proyecto (actualizado el 13 de agosto de 2026)

Esta serie conserva explicaciones y resultados históricos, pero la referencia
operativa actual es la siguiente:

- El sistema experto tiene **193 reglas**, CF estilo MYCIN, meta-reglas,
  encadenamiento hacia adelante y hacia atrás. Su voto usa OpenAI como
  proveedor principal, con heurísticas OpenCV que refinan atributos.
- El modelo local que participa en la decisión es **MobileNetV2 TFLite
  float32**, corrida `run_20260721_2129`; clasifica solo `plastico | vidrio`.
  MobileNetV3-Large INT8 está archivado como respaldo y **no emite votos**.
- En 1.000 capturas OV3660/QVGA, V2 obtuvo **71,60 %** de exactitud y
  **71,25 %** de macro-F1; V3 INT8 obtuvo 57,10 % y 57,09 %. La validación
  histórica de 98,43 % no describe por sí sola el rendimiento del robot.
- La ESP32-CAM toma tres fotos: se suman los seis votos válidos de ambas
  fuentes. `desconocido` es abstención; un empate se resuelve con el proveedor.
  Si el proveedor se abstiene las tres veces, el modelo local necesita 3/3.

La documentación operativa es [`ia/vision-service/README.md`](../../ia/vision-service/README.md),
[`model/README.md`](../../ia/vision-service/model/README.md) y
[`PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md`](../../docs/PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md).


## Encadenamiento hacia adelante

Es el modo natural: partir de los hechos y ver a dónde llevan.

```
hechos conocidos
     ↓
¿qué reglas tienen su antecedente satisfecho?
     ↓
dispararlas → acumular CF por categoría
     ↓
categoría con mayor CF
```

Se le llama *forward chaining* o razonamiento **dirigido por datos**. No hay
hipótesis previa: se aplica todo lo aplicable y se ve qué gana.

Con los hechos `objeto = botella_mocachino` y `confianza_ml = alta` dispara
`R01` (VIDRIO, 0,98). Si además dispararan otras dos reglas de vidrio, sus CF
se combinarían con la fórmula de MYCIN del documento anterior.



## Encadenamiento hacia atrás

Va al revés: parte de una **hipótesis** y busca si los hechos la sostienen.

```
hipótesis: "esto es VIDRIO"
     ↓
¿qué condiciones haría falta que se cumplieran?
     ↓
¿se cumplen en los hechos?
     ↓
puntaje de confirmación
```

En `backward_chaining.py` una hipótesis se representa como un `Goal`:

```python
Goal(categoria="VIDRIO",
     requisitos=[...],
     umbral_confianza=0.75)
```

El `umbral_confianza` es la **fracción de requisitos** que deben cumplirse para
dar la hipótesis por confirmada.

### Para qué sirve tener los dos

El forward dice "esto parece vidrio". El backward pregunta "si fuera vidrio,
¿tendría sentido todo lo demás que veo?". Se usa como **verificación cruzada**:
detecta cuando el forward llegó a una conclusión que el resto de la evidencia
contradice.

El motor arbitra con dos constantes (`inference_engine.py`):

```python
UMBRAL_BACKWARD   = 0.80   # score de backward que se considera contradicción
CF_FORWARD_SEGURO = 0.90   # si forward supera esto, se confía pese a backward
```

Se lee así: si el backward contradice con fuerza ≥ 0,80, se desconfía del
forward — **salvo** que el forward venga con CF ≥ 0,90, en cuyo caso manda él.
Es una jerarquía explícita entre dos formas de razonar, no un promedio.



## Meta-reglas

Aquí está la parte más inusual del diseño. Una **meta-regla** no clasifica
objetos: decide **cómo razonar**. Del encabezado de `meta_rules.py`:

> Reglas sobre cómo razonar — no clasifican objetos sino que controlan y
> ajustan el comportamiento del motor de inferencia.

Su estructura (`meta_rules.py:6`) es distinta a la de una regla normal:

```python
MetaRule(nombre, condicion, accion, descripcion, prioridad)
```

`condicion` y `accion` son **funciones**, no diccionarios de atributos. La
condición inspecciona los hechos y el contexto del razonamiento; la acción
modifica ese contexto. Y tienen `prioridad`, porque el orden en que se evalúan
importa.

La diferencia en una tabla:

| | Regla normal | Meta-regla |
| --- | --- | --- |
| Concluye sobre | El material del objeto | El proceso de razonamiento |
| Antecedente | Diccionario de atributos | Función sobre hechos y contexto |
| Efecto | Suma CF a una categoría | Cambia cómo sigue el motor |

Ejemplos del tipo de cosa que expresan: si los datos vienen con confianza baja,
exigir más evidencia; si dos categorías quedan empatadas, preferir la
abstención. Son las que permiten que el sistema se comporte distinto según la
**calidad** de lo que recibe, no solo según su contenido.



## Validación

Antes de razonar, `validator.py` comprueba que cada hecho tenga un valor de la
lista permitida ([03](03-sistema-experto-reglas.ipynb)).

No es un detalle menor. Los atributos llegan de un modelo de lenguaje que mira
la foto, y un modelo de lenguaje puede devolver `"color": "azulado"` cuando el
vocabulario solo admite ocho colores. Sin validación, el sistema razonaría sobre
un valor que ninguna regla puede satisfacer y se abstendría sin explicar por
qué. Con validación, el problema aparece nombrado.



## Explicación

`explanation.py` genera un reporte estructurado —exportable a JSON— con el
recorrido completo: hechos recibidos, reglas disparadas, CF acumulados,
resultado del backward, meta-reglas activadas.

De ahí sale el campo que viaja al robot:

```json
"rule_applied": "VIDRIO · 3 regla(s) · CF 0.95"
```

Es la propiedad que la red neuronal no tiene y la razón por la que el sistema
experto se conserva aunque un modelo entrenado pueda superarlo en exactitud:
cuando el robot abra la compuerta equivocada, esto es lo que permite averiguar
por qué.



## El recorrido completo

```
atributos del proveedor de visión
        ↓
   validación                    ¿son valores legales?
        ↓
   memoria de trabajo            hechos del objeto actual
        ↓
   meta-reglas                   ajustan cómo razonar
        ↓
   forward chaining              hechos → categorías, CF acumulado
        ↓
   backward chaining             ¿la evidencia sostiene la conclusión?
        ↓
   umbral 0,75                   ¿alcanza para abrir compuerta?
        ↓
   VIDRIO | PLASTICO | DESCONOCIDO  + reporte de explicación
```

Con esto queda cubierta la familia simbólica. Los siguientes tres documentos
tratan la otra.

---

← [Factor de certeza](04-factor-de-certeza.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Redes neuronales](06-redes-neuronales.ipynb) →
